In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

warnings.filterwarnings("ignore")

# ================================
# 1) AYARLAR
# ================================
preds_dir = "Preds"
manual_picks_file = "manual_picks.xlsx"
output_dir = "Performance_Results"
os.makedirs(output_dir, exist_ok=True)

# Zaman farkı eşiği (saniye) - sismolojik değerlendirme için ±0.5s veya ±3.0s yaygındır
MAX_TIME_DIFF = 100.0 

# ================================
# 2) YARDIMCI FONKSİYONLAR (Standardizasyon)
# ================================
def normalize_component(comp):
    """
    Sismik bileşenleri (Component) standart HZ, HN, HE formatına getirir.
    Örn: 'Z' -> 'HZ', 'HHZ' -> 'HZ', 'HZ' -> 'HZ'
    """
    comp = str(comp).strip().upper()
    
    # 1. Durum: Tek karakter (Z, N, E) -> Başına H ekle
    if len(comp) == 1:
        return 'H' + comp
    
    # 2. Durum: 3 karakter (HHZ, BHZ, HNZ) -> H + Son Harf
    if len(comp) == 3:
        return 'H' + comp[-1]
    
    # 3. Durum: Zaten 2 karakter (HZ, HN, HE) -> Aynen döndür
    return comp

# ================================
# 3) MANUEL PİCK'LERİ YÜKLE
# ================================
print("📂 Manuel pick'ler yükleniyor...")
manual_df = pd.read_excel(manual_picks_file)

# Kolon isimlerini normalize et
manual_df.columns = manual_df.columns.str.strip().str.lower()

# Gerekli kolonları kontrol et
required_cols = ["event_id", "station", "component", "phase", "manuel_pick"]
for col in required_cols:
    if col not in manual_df.columns:
        raise ValueError(f"❌ Manuel pick dosyasında '{col}' kolonu bulunamadı!")

# Component Standardizasyonu (Manuel veriyi de garantiye alalım)
manual_df['component'] = manual_df['component'].apply(normalize_component)

# Zaman ve Temizlik
manual_df["manuel_pick"] = pd.to_datetime(manual_df["manuel_pick"], errors="coerce")
if manual_df["manuel_pick"].dt.tz is not None:
    manual_df["manuel_pick"] = manual_df["manuel_pick"].dt.tz_localize(None)

manual_df = manual_df.dropna(subset=["event_id", "station", "component", "phase", "manuel_pick"])

# Event ID'leri string'e çevir (Matching için kritik)
manual_df['event_id'] = manual_df['event_id'].astype(str).str.strip()

print(f"✅ {len(manual_df)} manuel pick yüklendi")
print(f"   • Event sayısı: {manual_df['event_id'].nunique()}")

# ================================
# 4) MODEL SONUÇLARINI YÜKLE
# ================================
print("\n📂 Model sonuçları yükleniyor...")

csv_files = [f for f in os.listdir(preds_dir) if f.endswith('.csv') and 'performance' not in f]
model_results = {}

for csv_file in csv_files:
    parts = csv_file.replace('.csv', '').split('_')
    model_name = parts[0]
    # Dosya isimlendirmesine göre pretrained ismini çıkarma
    pretrained_name = '_'.join(parts[1:]) if len(parts) >= 2 else "unknown"
    model_key = f"{model_name}_{pretrained_name}"
    
    try:
        df = pd.read_csv(os.path.join(preds_dir, csv_file))
        
        # Gerekli kolon kontrolü
        if not all(col in df.columns for col in ["event_id", "station", "phase", "pick_time"]):
            continue
            
        # Component kolonu yoksa veya farklıysa kontrol et
        if "component" not in df.columns:
            # Component yoksa "channel"dan türetmeyi dene, o da yoksa atla
            if "channel" in df.columns:
                 df['component'] = df['channel'] # Geçici atama, aşağıda normalize edilecek
            else:
                print(f"⚠️ {csv_file}: Component/Channel bilgisi yok, atlanıyor.")
                continue

        # Component Standardizasyonu (KRİTİK ADIM: Z -> HZ dönüşümü burada yapılır)
        df['component'] = df['component'].apply(normalize_component)

        # Event ID Standardizasyonu
        df['event_id'] = df['event_id'].astype(str).str.strip()

        # Zaman formatı
        df["pick_time"] = pd.to_datetime(df["pick_time"], errors="coerce")
        if df["pick_time"].dt.tz is not None:
            df["pick_time"] = df["pick_time"].dt.tz_localize(None)
        
        df = df.dropna(subset=["event_id", "station", "component", "phase", "pick_time"])
        
        model_results[model_key] = {
            "data": df,
            "model": model_name,
            "pretrained": pretrained_name
        }
        print(f"✅ {model_key}: {len(df)} pick yüklendi.")
    
    except Exception as e:
        print(f"❌ {csv_file} hatası: {e}")

if not model_results:
    raise ValueError("❌ Hiç model sonucu yüklenemedi!")

# ================================
# 5) EŞLEŞTİRME VE ANALİZ (DÜZELTİLMİŞ MATCHING)
# ================================
print("\n🔗 Manuel ve model pick'leri eşleştiriliyor...")

def match_picks_robust(manual, model):
    """
    Robust eşleştirme fonksiyonu.
    """
    # Manuel DataFrame kopyası
    man_df = manual.copy()
    mod_df = model.copy()
    
    # Merge Key Oluşturma (Artık componentler eşitlendiği için eşleşecek)
    # Key Format: EVENTID_STATION_COMPONENT_PHASE (örn: 286369_DKL_HZ_P)
    man_df['merge_key'] = (
        man_df['event_id'] + '_' + 
        man_df['station'].str.upper() + '_' + 
        man_df['component'] + '_' + 
        man_df['phase'].str.upper()
    )
    
    mod_df['merge_key'] = (
        mod_df['event_id'] + '_' + 
        mod_df['station'].str.upper() + '_' + 
        mod_df['component'] + '_' + 
        mod_df['phase'].str.upper()
    )
    
    # Model tarafında duplicate varsa (aynı key için birden fazla tahmin),
    # en yüksek confidence'a sahip olanı tut.
    if 'confidence' in mod_df.columns:
        mod_df = mod_df.sort_values('confidence', ascending=False)
    
    mod_picks_unique = mod_df.drop_duplicates(subset=['merge_key'], keep='first')
    
    # Left Merge: Manuel picklerin hepsi kalır, eşleşen model pickleri yanına gelir
    matched = pd.merge(
        man_df[['merge_key', 'event_id', 'station', 'component', 'phase', 'manuel_pick']],
        mod_picks_unique[['merge_key', 'pick_time', 'confidence']] if 'confidence' in mod_picks_unique.columns else mod_picks_unique[['merge_key', 'pick_time']],
        on='merge_key',
        how='left'
    )
    
    matched = matched.rename(columns={'pick_time': 'model_pick'})
    
    # False Positives Hesaplama (Modelde var, Manuelde yok)
    # Modelin ürettiği unique key kümesi - Manuel key kümesi
    manual_keys_set = set(man_df['merge_key'])
    model_keys_set = set(mod_picks_unique['merge_key'])
    fp_keys = model_keys_set - manual_keys_set
    fp_count = len(fp_keys)
    
    return matched, fp_count

# --- Analiz Döngüsü ---
performance_summary = []
debug_once = True

for key, info in model_results.items():
    matched_df, fp_count = match_picks_robust(manual_df, info["data"])
    
    # DEBUG: İlk model için anahtarları kontrol et
    if debug_once:
        print("\n🔍 DEBUG (Düzeltilmiş Anahtarlar):")
        print("Manuel Key Örneği:", matched_df['merge_key'].iloc[0])
        # Model datasından örnek bir key oluşturup bakalım
        sample_model_row = info["data"].iloc[0]
        sample_model_key = f"{sample_model_row['event_id']}_{sample_model_row['station'].upper()}_{sample_model_row['component']}_{sample_model_row['phase'].upper()}"
        print("Model Key Örneği: ", sample_model_key)
        debug_once = False

    # Zaman Farkı Hesapla
    matched_df['time_diff'] = (matched_df['model_pick'] - matched_df['manuel_pick']).dt.total_seconds()
    matched_df['abs_diff'] = matched_df['time_diff'].abs()
    
    # Valid Match: Eşleşmiş VE zaman farkı tolerans (MAX_TIME_DIFF) içinde olanlar
    # Literatürde buna "True Positive" denir.
    tp_matches = matched_df[
        (matched_df['model_pick'].notna()) & 
        (matched_df['abs_diff'] <= MAX_TIME_DIFF)
    ]
    
    # Metrikler
    TP = len(tp_matches) # Başarılı tespit
    FN = len(matched_df) - len(matched_df[matched_df['model_pick'].notna()]) # Hiç tahmin yok (Miss)
    # Not: Zaman farkı > MAX_TIME_DIFF olanlar ne TP ne FN'dir, teknik olarak "Zamanlama Hatası" olan FP'dir.
    # Ancak basitlik için genellikle False Negative (kaçırılmış) veya False Positive (yanlış yerde) olarak kategorize edilir.
    # Burada katı bir değerlendirme yapalım:
    # TP: Tolerans içinde eşleşenler.
    # FN: Eşleşmeyenler (Model pick üretmedi).
    # FP: Modelin ürettiği ama manuelde olmayanlar (Extra) + Tolerans dışı olan eşleşmeler (Yanlış zaman).
    
    tolerance_failures = len(matched_df[matched_df['model_pick'].notna()]) - TP
    total_FP = fp_count + tolerance_failures
    
    # İstatistikler
    precision = TP / (TP + total_FP) if (TP + total_FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    mae = tp_matches['abs_diff'].mean()
    std = tp_matches['time_diff'].std()
    rmse = np.sqrt((tp_matches['time_diff'] ** 2).mean())
    
    performance_summary.append({
        'Model': info['model'],
        'Pretrained': info['pretrained'],
        'TP': TP,
        'FN': FN,
        'FP': total_FP,
        'Precision': precision,
        'Recall': recall,
        'F1_Score': f1,
        'MAE_sec': mae,
        'RMSE_sec': rmse,
        'Std_sec': std
    })
    
    print(f"\n📌 {key}:")
    print(f"   F1: {f1:.4f} | MAE: {mae:.4f}s | TP: {TP} / {len(manual_df)}")

# ================================
# 6) RAPORLAMA
# ================================
if performance_summary:
    res_df = pd.DataFrame(performance_summary)
    res_df = res_df.sort_values('MAE_sec', ascending=True) # En düşük hata en üstte
    
    print("\n" + "="*60)
    print("🏆 SONUÇ ÖZETİ (MAE Sıralı)")
    print("="*60)
    # Okunabilir format
    print(res_df[['Model', 'Pretrained', 'F1_Score', 'MAE_sec', 'TP', 'FP']].to_string(index=False, float_format="%.4f"))
    
    # CSV Kaydet
    res_df.to_csv(os.path.join(output_dir, "final_model_comparison.csv"), index=False)
    print(f"\n📄 Detaylı sonuçlar kaydedildi: {os.path.join(output_dir, 'final_model_comparison.csv')}")
    
    # Basit Bar Plot (MAE)
    plt.figure(figsize=(12, 6))
    sns.barplot(data=res_df, x='MAE_sec', y='Pretrained', hue='Model')
    plt.title(f'Model Timing Accuracy (MAE) - Lower is Better (Tolerance: {MAX_TIME_DIFF}s)')
    plt.xlabel('Mean Absolute Error (s)')
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "mae_comparison.png"), dpi=300)
    print("📈 Grafik kaydedildi.")

else:
    print("❌ Herhangi bir sonuç üretilemedi.")

In [ ]:

# ---- EXPORT model_results DICTIONARY AS CLEAN CSV ----
# concatenate dataframes stored in model_results and write to a CSV
all_dfs = []
for key, info in model_results.items():
    df_temp = info['data'].copy()
    df_temp['model'] = info['model']
    df_temp['pretrained'] = info['pretrained']
    all_dfs.append(df_temp)

if all_dfs:
    combined_results = pd.concat(all_dfs, ignore_index=True)
    combined_results.to_csv('model_results_clean.csv', index=False, encoding='utf-8-sig')
    print(f"✅ model_results dictionary exported to 'model_results_clean.csv' ({len(combined_results)} rows)")
else:
    print("⚠️ model_results dictionary is empty, nothing to export.")


In [ ]:
combined_results.model.value_counts()

In [ ]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 1. AYARLAR
# ==========================================
model_results_csv = "TUM_SONUCLAR_BIRLESIK_CLEAN.csv"
manual_picks_file = "manual_picks.xlsx"
output_folder = "Event_Bazli_Sonuclar_Hassas"  # Yeni klasör adı

# Hata Toleransı (Saniye)
MATCH_TOLERANCE = 100.0

print("="*80)
print("📂 DEPREM BAZLI AYRIŞTIRMA (Milisaniye Hassasiyetli)")
print("="*80)

os.makedirs(output_folder, exist_ok=True)

# ==========================================
# 2. YARDIMCI FONKSİYONLAR
# ==========================================
def normalize_component(comp):
    if pd.isna(comp): return np.nan
    comp = str(comp).strip().upper()
    if len(comp) == 1: return 'H' + comp
    if len(comp) == 3: return 'H' + comp[-1]
    return comp

def clean_event_id(id_val):
    if pd.isna(id_val): return ""
    return str(id_val).split('.')[0].strip()

# ==========================================
# 3. VERİLERİ YÜKLE
# ==========================================
print("\n1️⃣ Veriler yükleniyor...")

if not os.path.exists(manual_picks_file) or not os.path.exists(model_results_csv):
    raise FileNotFoundError("❌ Girdi dosyaları bulunamadı.")

# --- Manuel Veri ---
df_manual = pd.read_excel(manual_picks_file)
df_manual.columns = df_manual.columns.str.strip().str.lower()

df_manual['event_id'] = df_manual['event_id'].apply(clean_event_id)
df_manual['station'] = df_manual['station'].astype(str).str.upper().str.strip()
df_manual['phase'] = df_manual['phase'].astype(str).str.upper().str.strip()
df_manual['component'] = df_manual['component'].apply(normalize_component)
df_manual['manuel_pick'] = pd.to_datetime(df_manual['manuel_pick'], errors='coerce').dt.tz_localize(None)

df_manual_clean = df_manual[['event_id', 'station', 'component', 'phase', 'manuel_pick']].rename(
    columns={'manuel_pick': 'manual_pick_time'}
)

# --- Model Verisi ---
df_model = pd.read_csv(model_results_csv, dtype={'confidence': float})

df_model['event_id'] = df_model['event_id'].apply(clean_event_id)
df_model['station'] = df_model['station'].astype(str).str.upper().str.strip()
df_model['phase'] = df_model['phase'].astype(str).str.upper().str.strip()
if 'component' in df_model.columns:
    df_model['component'] = df_model['component'].apply(normalize_component)

df_model['pick_time'] = pd.to_datetime(df_model['pick_time'], errors='coerce').dt.tz_localize(None)

print(f"✅ {len(df_manual_clean)} manuel, {len(df_model)} model tahmini hazır.")

# ==========================================
# 4. EŞLEŞTİRME VE HESAPLAMA
# ==========================================
print("\n2️⃣ Eşleştirme ve hesaplama yapılıyor...")

# Tek seferde Merge (EventID + Station + Comp + Phase)
merged_total = pd.merge(
    df_model,
    df_manual_clean,
    on=['event_id', 'station', 'component', 'phase'],
    how='left'
)

# Fark Hesapla (Saniye)
merged_total['raw_diff'] = (merged_total['pick_time'] - merged_total['manual_pick_time']).dt.total_seconds()

# Tolerans Filtresi
outlier_mask = merged_total['raw_diff'].abs() > MATCH_TOLERANCE
merged_total.loc[outlier_mask, 'manual_pick_time'] = pd.NaT
merged_total.loc[outlier_mask, 'error_sec'] = None 

valid_mask = ~outlier_mask & merged_total['raw_diff'].notna()
merged_total.loc[valid_mask, 'error_sec'] = merged_total.loc[valid_mask, 'raw_diff'].round(4) # 4 hane hassasiyet

# Excel Format Koruması (Hata Sütunu)
if 'error_sec' in merged_total.columns:
    merged_total['error_sec'] = merged_total['error_sec'].astype(str).replace({'nan': '', 'None': ''})

# ==========================================
# 5. SALİSELERİ GARANTİLEME (FORMATLAMA)
# ==========================================
print("3️⃣ Zaman formatları (milisaniye) ayarlanıyor...")

# Bu adım, datetime objelerini, milisaniyeleri içeren STRİNGLERE dönüştürür.
# Format: YYYY-MM-DD HH:MM:SS.mmmmmm
# Örn: 2015-01-23 10:17:59.123456

if 'pick_time' in merged_total.columns:
    merged_total['pick_time'] = merged_total['pick_time'].dt.strftime('%Y-%m-%d %H:%M:%S.%f')

if 'manual_pick_time' in merged_total.columns:
    merged_total['manual_pick_time'] = merged_total['manual_pick_time'].dt.strftime('%Y-%m-%d %H:%M:%S.%f')

# NaT olanlar (eşleşmeyenler) "NaT" stringi olarak kalabilir veya boşaltılabilir.
# Temiz görünüm için boşaltalım:
merged_total['pick_time'] = merged_total['pick_time'].fillna('')
merged_total['manual_pick_time'] = merged_total['manual_pick_time'].fillna('')

# ==========================================
# 6. KAYDETME
# ==========================================
print("4️⃣ Dosyalar kaydediliyor...")

cols = ['event_id', 'station', 'component', 'phase', 'model', 'pretrained', 
        'pick_time', 'manual_pick_time', 'error_sec', 'confidence']
remaining = [c for c in merged_total.columns if c not in cols and c != 'raw_diff']
final_columns = cols + remaining

grouped = merged_total.groupby('event_id')

for event_id, group_df in tqdm(grouped):
    # Sıralama
    group_df = group_df.sort_values(by=['station', 'phase', 'confidence'], ascending=[True, True, False])
    
    output_df = group_df[final_columns]
    
    out_file = os.path.join(output_folder, f"Event_{event_id}.csv")
    # index=False, encoding='utf-8-sig' önemlidir
    output_df.to_csv(out_file, index=False, encoding='utf-8-sig')

print("\n" + "="*80)
print("✅ İŞLEM TAMAMLANDI!")
print(f"📂 Sonuçlar: {output_folder}")
print(f"ℹ️  Zamanlar '%Y-%m-%d %H:%M:%S.%f' formatında metin olarak kaydedildi.")
print("="*80)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

# Akademik yayın standardı: Gereksiz uyarıları bastır
warnings.filterwarnings("ignore")

# ================================
# 1) AYARLAR
# ================================
preds_dir = "Preds"
manual_picks_file = "manual_picks.xlsx"
output_dir = "Performance_Results_100s"  # Çıktı klasörünü değiştirdim
os.makedirs(output_dir, exist_ok=True)

# ⚡ KRİTİK DEĞİŞİKLİK: Tolerans 100.0 Saniye Yapıldı
MATCH_TOLERANCE = 100.0 

print(f"📊 ANALİZ BAŞLATILIYOR (Tolerans: ±{MATCH_TOLERANCE} saniye)")
print("="*60)

# ================================
# 2) YARDIMCI FONKSİYONLAR (Standardizasyon)
# ================================
def normalize_component(comp):
    """Bileşenleri standart HZ, HN, HE formatına getirir."""
    if pd.isna(comp): return np.nan
    comp = str(comp).strip().upper()
    if len(comp) == 1: return 'H' + comp      # Z -> HZ
    if len(comp) == 3: return 'H' + comp[-1]  # HHZ -> HZ
    return comp

def clean_event_id(id_val):
    """Event ID temizliği (Excel int/str karmaşasını önler)"""
    if pd.isna(id_val): return ""
    return str(id_val).split('.')[0].strip()

# ================================
# 3) VERİ YÜKLEME
# ================================
print("1️⃣  Manuel pick'ler yükleniyor...")
df_manual = pd.read_excel(manual_picks_file)
df_manual.columns = df_manual.columns.str.strip().str.lower()

# Veri Temizliği
df_manual['event_id'] = df_manual['event_id'].apply(clean_event_id)
df_manual['station'] = df_manual['station'].astype(str).str.upper().str.strip()
df_manual['phase'] = df_manual['phase'].astype(str).str.upper().str.strip()
df_manual['component'] = df_manual['component'].apply(normalize_component)
df_manual['manuel_pick'] = pd.to_datetime(df_manual['manuel_pick'], errors='coerce').dt.tz_localize(None)

# Temiz Manuel Set
df_manual_clean = df_manual[['event_id', 'station', 'component', 'phase', 'manuel_pick']].rename(
    columns={'manuel_pick': 'manual_pick_time'}
)
print(f"   ✅ {len(df_manual_clean)} manuel pick hazır.")

print("\n2️⃣  Model sonuçları yükleniyor...")
# Burada tek bir büyük dosya okuyorsak:
if os.path.exists("TUM_SONUCLAR_BIRLESIK_CLEAN.csv"):
    df_model = pd.read_csv("TUM_SONUCLAR_BIRLESIK_CLEAN.csv", dtype={'confidence': float})
else:
    raise FileNotFoundError("❌ Model sonuç dosyası bulunamadı.")

# Model Veri Temizliği
df_model['event_id'] = df_model['event_id'].apply(clean_event_id)
df_model['station'] = df_model['station'].astype(str).str.upper().str.strip()
df_model['phase'] = df_model['phase'].astype(str).str.upper().str.strip()
if 'component' in df_model.columns:
    df_model['component'] = df_model['component'].apply(normalize_component)
df_model['pick_time'] = pd.to_datetime(df_model['pick_time'], errors='coerce').dt.tz_localize(None)

print(f"   ✅ {len(df_model)} model tahmini hazır.")

# ================================
# 4) EŞLEŞTİRME VE ANALİZ (ROBUST MATCHING)
# ================================
print("\n3️⃣  Eşleştirme ve Performans Hesabı...")

# Model bazlı gruplama (Her model için ayrı istatistik)
model_stats = []
grouped_models = df_model.groupby(['model', 'pretrained'])

for (model_name, pretrained), group_df in tqdm(grouped_models):
    
    # A. KESİN EŞLEŞTİRME (Merge Key: EventID + Station + Component + Phase)
    # Zaman farkına bakmaksızın önce mantıksal eşleşmeyi buluyoruz.
    merged = pd.merge(
        df_manual_clean,
        group_df,
        on=['event_id', 'station', 'component', 'phase'],
        how='left' # Tüm manuel pickleri tut (False Negative hesabı için)
    )
    
    # B. ZAMAN FARKI HESABI
    # Saniye cinsinden fark (Model - Manuel)
    merged['time_diff'] = (merged['pick_time'] - merged['manual_pick_time']).dt.total_seconds()
    merged['abs_diff'] = merged['time_diff'].abs()
    
    # C. TOLERANS FİLTRESİ (100 saniye)
    # True Positive (TP): Eşleşme var VE fark <= 100s
    tp_matches = merged[
        (merged['pick_time'].notna()) & 
        (merged['abs_diff'] <= MATCH_TOLERANCE)
    ]
    
    # İstatistikler
    TP = len(tp_matches)
    FN = len(merged) - len(merged[merged['pick_time'].notna()]) # Hiç tahmin yok
    
    # False Positive (FP) Hesabı:
    # 1. Modelin ürettiği toplam tahmin sayısı (group_df)
    # 2. Bunlardan TP olanları çıkarırsak, geriye "Yanlış" veya "Tolerans Dışı" olanlar kalır.
    total_predictions = len(group_df)
    FP = total_predictions - TP 
    
    # Metrikler
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    mae = tp_matches['abs_diff'].mean()
    std = tp_matches['time_diff'].std()
    
    model_stats.append({
        'Model': model_name,
        'Pretrained': pretrained,
        'TP': TP, 'FN': FN, 'FP': FP,
        'Precision': precision, 'Recall': recall, 'F1_Score': f1,
        'MAE_sec': mae, 'Std_sec': std
    })

# ================================
# 5) SONUÇLARI RAPORLA
# ================================
df_results = pd.DataFrame(model_stats).sort_values('MAE_sec', ascending=True)

print("\n" + "="*60)
print(f"🏆 SONUÇLAR (Tolerans: {MATCH_TOLERANCE}s, MAE Sıralı)")
print("="*60)
print(df_results[['Model', 'Pretrained', 'F1_Score', 'MAE_sec', 'TP', 'FP']].to_string(index=False, float_format="%.4f"))

# CSV Kaydet
csv_path = os.path.join(output_dir, "model_performance_100s.csv")
df_results.to_csv(csv_path, index=False)
print(f"\n📄 Sonuçlar kaydedildi: {csv_path}")

# GRAFİK: MAE Karşılaştırması
plt.figure(figsize=(12, 6))
sns.barplot(data=df_results, x='MAE_sec', y='Pretrained', hue='Model')
plt.title(f'Model Timing Accuracy (MAE) - Tolerance: {MATCH_TOLERANCE}s')
plt.xlabel('Mean Absolute Error (seconds)')
plt.tight_layout()
plt.savefig(os.path.join(output_dir, "mae_comparison_100s.png"), dpi=300)
print("📈 Grafik kaydedildi.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

# ==========================================
# 1. AKADEMİK GÖRSEL AYARLARI (PUBLICATION-READY)
# ==========================================
# IEEE/ACM standartlarına uygun font ve stil ayarları
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif'],
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.dpi': 300,          # Yüksek çözünürlük
    'savefig.dpi': 300,
    'savefig.format': 'pdf',    # Vektörel çıktı (PDF/EPS önerilir)
    'savefig.bbox': 'tight'
})

OUTPUT_DIR = "Analiz_Raporu"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# 2. VERİ YÜKLEME (Önceki analizden gelen CSV'ler)
# ==========================================
# Not: Bu dosyaların çalışma dizininde olduğu varsayılmıştır.
try:
    df_ranking = pd.read_csv(os.path.join(OUTPUT_DIR, "Model_Ranking.csv"))
    df_station = pd.read_csv(os.path.join(OUTPUT_DIR, "Station_Performance.csv"))
    # Magnitüd analizi için detaylı veri setine ihtiyaç vardır (önceki koddan df_best veya df_analysis)
    # Eğer dosya yoksa simülasyon yapılır (Kodun çalışması için)
except FileNotFoundError:
    print("⚠️ Gerekli CSV dosyaları bulunamadı. Lütfen önce analiz kodunu çalıştırın.")
    # Demo amaçlı boş dataframe oluşturulabilir veya script durdurulabilir.

# ==========================================
# ŞEKİL 1: Model Performans Haritası (Scatter Plot)
# ==========================================
def plot_model_comparison(df):
    plt.figure(figsize=(10, 6))
    
    # Scatter plot: X=Recall, Y=Precision, Boyut=F1, Renk=Model
    scatter = sns.scatterplot(
        data=df, 
        x='Recall', 
        y='Precision', 
        size='F1_Score', 
        hue='Model',
        sizes=(100, 600), 
        alpha=0.8,
        edgecolor='k',
        palette='viridis'
    )
    
    # Her noktanın yanına 'Pretrained' ismini yaz
    for i in range(df.shape[0]):
        plt.text(
            df.Recall.iloc[i]+0.002, 
            df.Precision.iloc[i]+0.002, 
            df.Pretrained.iloc[i], 
            horizontalalignment='left', 
            size='small', 
            color='black',
            weight='semibold'
        )

    # İdeal bölgeyi gösteren ok (Opsiyonel)
    plt.annotate('İdeal Bölge', xy=(0.98, 0.98), xytext=(0.90, 0.90),
                 arrowprops=dict(facecolor='black', shrink=0.05))

    plt.title('Model Performans Karşılaştırması (Precision vs. Recall)', fontweight='bold')
    plt.xlabel('Duyarlılık (Recall)')
    plt.ylabel('Kesinlik (Precision)')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)
    
    # Kaydet
    plt.savefig(os.path.join(OUTPUT_DIR, "Fig1_Model_Comparison.pdf"))
    plt.savefig(os.path.join(OUTPUT_DIR, "Fig1_Model_Comparison.png")) # Raster yedeği
    plt.show()
    print("✅ Şekil 1 oluşturuldu.")

# ==========================================
# ŞEKİL 2: Magnitüd vs. Zamanlama Hatası (Regresyon)
# ==========================================
def plot_magnitude_error(df_detail, model_name):
    """
    df_detail: 'magnitude' ve 'abs_error' sütunlarını içermelidir.
    """
    plt.figure(figsize=(8, 6))
    
    # Hexbin veya Scatter kullanılabilir. Veri yoğunsa hexbin, azsa scatter.
    # Regresyon çizgisi ekle
    sns.regplot(
        data=df_detail, 
        x='magnitude', 
        y='abs_error', 
        scatter_kws={'alpha': 0.3, 's': 10}, 
        line_kws={'color': 'red', 'linewidth': 2},
        ci=95 # %95 Güven Aralığı
    )
    
    # İstatistiksel açıklama (Pearson Korelasyonu)
    # Not: NaN değerleri temizlenmeli
    clean_data = df_detail.dropna(subset=['magnitude', 'abs_error'])
    r, p = stats.pearsonr(clean_data['magnitude'], clean_data['abs_error'])
    
    plt.text(0.05, 0.9, f"$r = {r:.2f}$\n$p < 0.001$", transform=plt.gca().transAxes, 
             bbox=dict(facecolor='white', alpha=0.8))

    plt.title(f'Deprem Büyüklüğü ile Zamanlama Hatası İlişkisi\n({model_name})', fontweight='bold')
    plt.xlabel('Deprem Büyüklüğü ($M_w$)')
    plt.ylabel('Mutlak Zamanlama Hatası (s)')
    plt.ylim(0, 1.0) # Odaklanmak için Y eksenini sınırla
    
    plt.savefig(os.path.join(OUTPUT_DIR, "Fig2_Magnitude_Error.pdf"))
    plt.show()
    print("✅ Şekil 2 oluşturuldu.")

# ==========================================
# ŞEKİL 3: İstasyon Hata Dağılımı (Histogram)
# ==========================================
def plot_station_distribution(df_station):
    plt.figure(figsize=(10, 6))
    
    # Histogram ve KDE (Kernel Density Estimation)
    sns.histplot(
        df_station['MAE'], 
        bins=30, 
        kde=True, 
        color='steelblue', 
        edgecolor='white',
        line_kws={'linewidth': 2}
    )
    
    # Ortalama çizgisi
    mean_val = df_station['MAE'].mean()
    plt.axvline(mean_val, color='red', linestyle='--', label=f'Ortalama: {mean_val:.3f}s')
    
    plt.title('İstasyon Bazlı Ortalama Mutlak Hata (MAE) Dağılımı', fontweight='bold')
    plt.xlabel('MAE (saniye)')
    plt.ylabel('İstasyon Sayısı (Frekans)')
    plt.legend()
    
    plt.savefig(os.path.join(OUTPUT_DIR, "Fig3_Station_Distribution.pdf"))
    plt.show()
    print("✅ Şekil 3 oluşturuldu.")

# ==========================================
# TABLO ÜRETİMİ (LaTeX Formatı)
# ==========================================
def generate_latex_tables(df_ranking, df_station):
    # 1. Model Sıralama Tablosu (Top 5)
    print("\n--- LaTeX Tablo Kodu (Model Sıralaması) ---")
    top_models = df_ranking.head(5)[['Model', 'Pretrained', 'F1_Score', 'MAE_sec', 'Precision', 'Recall']]
    latex_code = top_models.to_latex(
        index=False, 
        float_format="%.3f",
        caption="Derin öğrenme modellerinin performans karşılaştırması.",
        label="tab:model_performance",
        column_format="llcccc"
    )
    print(latex_code)
    
    # 2. Sorunlu İstasyonlar Tablosu (En yüksek hataya sahip 5 istasyon)
    print("\n--- LaTeX Tablo Kodu (Sorunlu İstasyonlar) ---")
    worst_stations = df_station.sort_values('MAE', ascending=False).head(5)
    latex_station = worst_stations.to_latex(
        index=True, # İstasyon adları index ise
        float_format="%.3f",
        caption="En yüksek ortalama hataya sahip istasyonlar.",
        label="tab:worst_stations"
    )
    print(latex_station)

# ==========================================
# ÇALIŞTIRMA
# ==========================================
if 'df_ranking' in locals():
    plot_model_comparison(df_ranking)

# Not: df_best verisi önceki analizden gelmelidir. 
# Eğer mevcutsa aşağıdaki satırı aktif edin:
# plot_magnitude_error(df_best, "En İyi Model") 

if 'df_station' in locals():
    plot_station_distribution(df_station)
    generate_latex_tables(df_ranking, df_station)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import os

# ==========================================
# 1. GÖRSEL AYARLARI
# ==========================================
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.figsize': (10, 6),
    'figure.dpi': 150,          
    'savefig.dpi': 300,
    'savefig.format': 'png',    # Markdown için PNG daha iyidir
    'savefig.bbox': 'tight'
})

OUTPUT_DIR = "Analiz_Raporu"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# 2. VERİ YÜKLEME
# ==========================================
# Dosyaların varlığını kontrol et (Kodun çalışması için örnek veri simülasyonu yoktur)
try:
    df_ranking = pd.read_csv(os.path.join(OUTPUT_DIR, "Model_Ranking.csv"))
    df_station = pd.read_csv(os.path.join(OUTPUT_DIR, "Station_Performance.csv"))
    print("✅ Veriler yüklendi.")
except FileNotFoundError:
    print("⚠️ CSV dosyaları bulunamadı. Lütfen önce analiz kodunu çalıştırın.")

# ==========================================
# ŞEKİL 1: Model Performans Haritası
# ==========================================
def plot_model_comparison(df):
    plt.figure(figsize=(10, 6))
    sns.scatterplot(
        data=df, x='Recall', y='Precision', size='F1_Score', 
        hue='Model', sizes=(100, 500), alpha=0.7, edgecolor='black'
    )
    
    # Etiketler
    for i in range(df.shape[0]):
        plt.text(
            df.Recall.iloc[i]+0.001, df.Precision.iloc[i]+0.001, 
            df.Pretrained.iloc[i], size='small', color='black'
        )

    plt.title('Model Performans Karşılaştırması')
    plt.xlabel('Recall (Duyarlılık)')
    plt.ylabel('Precision (Kesinlik)')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    
    save_path = os.path.join(OUTPUT_DIR, "Fig1_Model_Comparison.png")
    plt.savefig(save_path)
    plt.show()
    print(f"✅ Grafik kaydedildi: {save_path}")

# ==========================================
# ŞEKİL 2: Magnitüd Analizi
# ==========================================
def plot_magnitude_error(df_detail):
    # Not: Bu fonksiyon için df_detail verisine ihtiyaç vardır
    plt.figure(figsize=(8, 6))
    sns.regplot(data=df_detail, x='magnitude', y='abs_error', 
                scatter_kws={'alpha':0.3}, line_kws={'color':'red'})
    
    plt.title('Deprem Büyüklüğü vs. Hata')
    plt.xlabel('Magnitüd')
    plt.ylabel('Hata (sn)')
    plt.ylim(0, 1.0)
    
    save_path = os.path.join(OUTPUT_DIR, "Fig2_Magnitude_Error.png")
    plt.savefig(save_path)
    plt.show()
    print(f"✅ Grafik kaydedildi: {save_path}")

# ==========================================
# ŞEKİL 3: İstasyon Dağılımı
# ==========================================
def plot_station_distribution(df_st):
    plt.figure(figsize=(10, 6))
    sns.histplot(df_st['MAE'], bins=20, kde=True, color='teal')
    plt.axvline(df_st['MAE'].mean(), color='red', linestyle='--', label='Ortalama')
    
    plt.title('İstasyon Hata Dağılımı')
    plt.xlabel('MAE (saniye)')
    plt.ylabel('İstasyon Sayısı')
    plt.legend()
    
    save_path = os.path.join(OUTPUT_DIR, "Fig3_Station_Distribution.png")
    plt.savefig(save_path)
    plt.show()
    print(f"✅ Grafik kaydedildi: {save_path}")

# ==========================================
# MARKDOWN TABLO ÜRETİCİSİ
# ==========================================
def print_markdown_tables(df_rank, df_st):
    print("\n" + "="*40)
    print("📌 RAPORA YAPIŞTIRILACAK TABLOLAR")
    print("="*40)
    
    # 1. Model Sıralaması
    print("\n### Tablo 1: Model Sıralaması (Kopyala-Yapıştır)")
    top_models = df_rank.head(10)[['Model', 'Pretrained', 'F1_Score', 'MAE_sec', 'Precision', 'Recall']]
    # Markdown formatında yazdır
    print(top_models.to_markdown(index=False, floatfmt=".4f"))
    
    # 2. Sorunlu İstasyonlar
    print("\n### Tablo 2: En Yüksek Hatalı İstasyonlar (Kopyala-Yapıştır)")
    worst_stations = df_st.sort_values('MAE', ascending=False).head(5)
    print(worst_stations.to_markdown(index=True, floatfmt=".4f"))
    print("\n" + "="*40)

# ==========================================
# ÇALIŞTIRMA
# ==========================================
if 'df_ranking' in locals():
    plot_model_comparison(df_ranking)
    
if 'df_station' in locals():
    plot_station_distribution(df_station)
    print_markdown_tables(df_ranking, df_station)

In [ ]:
import pandas as pd

# Dosya yolu
file_path = "manual_picks.xlsx"

# Veriyi yükle
df = pd.read_excel(file_path)

# Veri temizliği (Boşlukları sil ve büyük harfe çevir ki 'GULT' ile 'gult ' aynı sayılsın)
df['station'] = df['station'].astype(str).str.strip().str.upper()
df['event_id'] = df['event_id'].astype(str).str.strip()

# Sayıları hesapla
num_stations = df['station'].nunique()
num_events = df['event_id'].nunique()

print("="*40)
print(f"📡 İstasyon Sayısı: {num_stations}")
print(f"🌍 Deprem Sayısı:   {num_events}")
print("="*40)

In [ ]:

# --------------------------------------------------
# Ek: Tüm modelleri içeren ikinci bir CSV oluştur ve dışarı aktar
# --------------------------------------------------
print("\n🔁 Ek CSV oluşturuluyor...")

preds_dir = "Preds"
output_csv2 = "TUM_SONUCLAR_BIRLESIK_ALLMODELS.csv"

# Önceki birleştirme kodu tekrar kullanılabilir
csv_files = [f for f in os.listdir(preds_dir) if f.endswith('.csv') and 'performance' not in f]

if os.path.exists(output_csv2):
    os.remove(output_csv2)

for i, csv_file in enumerate(csv_files):
    path = os.path.join(preds_dir, csv_file)
    try:
        df = pd.read_csv(path)
        # Normalize / clean same şekilde
        if 'component' in df.columns:
            df['component'] = df['component'].apply(normalize_component)
        if 'event_id' in df.columns:
            df['event_id'] = df['event_id'].astype(str).str.strip()
        if 'pick_time' in df.columns:
            df['pick_time'] = pd.to_datetime(df['pick_time'], errors='coerce')
            if df['pick_time'].dt.tz is not None:
                df['pick_time'] = df['pick_time'].dt.tz_localize(None)
        # drop duplicates
        key_cols = ['event_id','station','component','phase']
        existing = [c for c in key_cols if c in df.columns]
        if existing:
            if 'confidence' in df.columns:
                df = df.sort_values('confidence', ascending=False)
            df = df.drop_duplicates(subset=existing, keep='first')
        # append
        if i == 0:
            df.to_csv(output_csv2, index=False, encoding='utf-8-sig', mode='w')
        else:
            df.to_csv(output_csv2, index=False, encoding='utf-8-sig', mode='a', header=False)
    except Exception as e:
        print(f"⚠️ Hata ({csv_file}): {e}")

print(f"✅ Ek CSV tamamlandı: {os.path.abspath(output_csv2)}")
# Gösterim için ilk birkaç satır
try:
    tmp = pd.read_csv(output_csv2, nrows=5)
    print(tmp.head())
except:
    pass


In [ ]:
# ---- EK HATA AYIKLAMA: Preds klasöründeki CSV dosyaları inceleniyor ----
import os
import pandas as pd

# değişkenleri yeniden tanımla (bağımlılık olmaması için)
preds_dir = "Preds"
required = ["event_id", "station", "phase", "pick_time"]

# Liste dosyalar
all_files = [f for f in os.listdir(preds_dir) if f.endswith('.csv') and 'performance' not in f]
summary_lines = []
summary_lines.append(f"📂 Toplam {len(all_files)} CSV bulundu.")

# Okuma testleri
model_results = {}
skipped = []
for f in sorted(all_files):
    path = os.path.join(preds_dir, f)
    try:
        df = pd.read_csv(path)
        miss = [c for c in required if c not in df.columns]
        if miss:
            skipped.append((f, miss))
            continue
        model_results[f] = df.shape
    except Exception as e:
        skipped.append((f, str(e)))

summary_lines.append(f"\n✅ Başarılı olarak okunan dosyalar: {len(model_results)}")
# only add first 10 entries to avoid huge output
for i,(k,v) in enumerate(model_results.items()):
    if i>=10: break
    summary_lines.append(f" - {k}: {v[0]} satır, {v[1]} sütun")
if len(model_results)>10:
    summary_lines.append(f"   ... +{len(model_results)-10} diğer dosya")

if skipped:
    summary_lines.append(f"\n⚠️ Atlanan/hataya uğrayan dosya sayısı: {len(skipped)}")
    for item in skipped[:10]:
        summary_lines.append(f" - {item}")
    if len(skipped)>10:
        summary_lines.append(f"   ... +{len(skipped)-10} diğer")

summary_lines.append(f"\n🔑 Model anahtar sayısı: {len(model_results)}")
summary_lines.append(str(sorted(model_results.keys())))

# write to file and display
with open('preds_summary.txt','w',encoding='utf-8') as f:
    f.write("\n".join(summary_lines))

print('\n'.join(summary_lines))


In [ ]:
# ---- EXPORT ALL RESULTS ----
# Bu hücre, analiz sırasında oluşturulan tüm çıktıları tek bir ZIP dosyasına
# sıkıştırarak dışarı aktarır. İhtiyacınıza göre dizinleri düzenleyebilirsiniz.
import os
import glob
from zipfile import ZipFile

result_dirs = [
    "Performance_Results",
    "Performance_Results_100s",
    "Event_Bazli_Sonuclar_Hassas",
    "Analiz_Raporu"
]
zip_path = "all_results.zip"

with ZipFile(zip_path, 'w') as z:
    for d in result_dirs:
        if not os.path.isdir(d):
            continue
        for filepath in glob.glob(os.path.join(d, "**"), recursive=True):
            if os.path.isfile(filepath):
                # arcname ile zip içinde kök dizin ismini kaldır
                arcname = os.path.relpath(filepath, start=os.path.dirname(d))
                z.write(filepath, arcname)

print(f"✅ Tüm sonuçlar '{zip_path}' olarak paketlendi.")
print("📁 İçerik listesi:")
with ZipFile(zip_path) as z:
    for name in z.namelist():
        print(" -", name)


In [ ]:
# ---- CREATE CLEAN ALL RESULTS CSV ----
# This cell concatenates all CSV results from the various output folders
# and writes a single cleaned CSV named 'clean_all_results.csv'.
import glob
import pandas as pd

out_dirs = [
    "Performance_Results",
    "Performance_Results_100s",
    "Event_Bazli_Sonuclar_Hassas",
    "Analiz_Raporu"
]
all_frames = []
for d in out_dirs:
    for f in glob.glob(os.path.join(d, "*.csv")):
        try:
            df = pd.read_csv(f)
            df['source_file'] = os.path.basename(f)
            all_frames.append(df)
        except Exception as e:
            print(f"⚠️ Okunamadı: {f} - {e}")

if all_frames:
    combined = pd.concat(all_frames, ignore_index=True)
    # drop duplicates if any
    combined = combined.drop_duplicates()
    combined.to_csv("clean_all_results.csv", index=False, encoding='utf-8-sig')
    print(f"✅ Tüm sonuçlar 'clean_all_results.csv' olarak kaydedildi ({len(combined)} satır).")
else:
    print("⚠️ Hiç CSV bulunamadı, combined dosya oluşturulamadı.")


In [ ]:
# ---- CHECK COLUMNS IN CLEAN CSV ----
import pandas as pd

try:
    df_clean = pd.read_csv("clean_all_results.csv")
    print("Columns:", df_clean.columns.tolist())
    print("Sample rows:")
    print(df_clean.head(3))
except Exception as e:
    print("Could not read clean_all_results.csv:", e)


In [ ]:
# ---- NORMALIZE COLUMN NAMES IN CLEAN CSV ----
# After initial creation we standardize names to match desired schema
import pandas as pd

try:
    df = pd.read_csv("clean_all_results.csv")
    # if both "Model" and "model" appear, drop lowercase version
    cols = df.columns.tolist()
    for name in ['model','pretrained']:
        if name in cols and name.title() in cols:
            df = df.drop(columns=[name])
            cols.remove(name)
    # ensure uppercase titles exist if only lowercase present
    if 'model' in df.columns and 'Model' not in df.columns:
        df = df.rename(columns={'model':'Model'})
    if 'pretrained' in df.columns and 'Pretrained' not in df.columns:
        df = df.rename(columns={'pretrained':'Pretrained'})
    # front-order the two columns
    cols = list(df.columns)
    front = [c for c in ['Model','Pretrained'] if c in cols]
    remaining = [c for c in cols if c not in front]
    df = df[front+remaining]
    df.to_csv("clean_all_results.csv", index=False, encoding='utf-8-sig')
    print("✅ Sütun isimleri standardize edildi. Güncel kolonlar:", df.columns.tolist())
except Exception as e:
    print("Kolonları normalizrken hata:", e)


In [ ]:
# ---- REBUILD CLEAN CSV WITH PROPER MODEL COLUMNS ----
# If model/pretrained columns disappeared, recreate the combined file correctly.
import glob
import pandas as pd

out_dirs = [
    "Performance_Results",
    "Performance_Results_100s",
    "Event_Bazli_Sonuclar_Hassas",
    "Analiz_Raporu"
]
frames = []
for d in out_dirs:
    for f in glob.glob(os.path.join(d, "*.csv")):
        try:
            tmp = pd.read_csv(f)
            # normalize column names for this df before concat
            if 'model' in tmp.columns and 'Model' not in tmp.columns:
                tmp = tmp.rename(columns={'model':'Model'})
            if 'pretrained' in tmp.columns and 'Pretrained' not in tmp.columns:
                tmp = tmp.rename(columns={'pretrained':'Pretrained'})
            # drop lowercase if both present
            for col in ['model','pretrained']:
                if col in tmp.columns and col.title() in tmp.columns:
                    tmp = tmp.drop(columns=[col])
            tmp['source_file'] = os.path.basename(f)
            frames.append(tmp)
        except Exception as e:
            print("⚠️ read error", f, e)
if frames:
    combined = pd.concat(frames, ignore_index=True)
    combined = combined.drop_duplicates()
    combined.to_csv("clean_all_results.csv", index=False, encoding='utf-8-sig')
    print("✅ Rebuilt clean_all_results.csv with", len(combined.columns), "columns and", len(combined), "rows.")
    print("Columns now:", combined.columns.tolist())
else:
    print("⚠️ no frames to combine")
